# 유저 탈퇴 분석

- 유저 탈퇴 테이블에는 **탈퇴 사유와 탈퇴 일시만 기록**되어 있어, 이탈한 사용자의 개별 특성이나 서비스 이용 행태를 직접적으로 파악하기에는 한계가 있습니다.

- 따라서 본 분석에서는 기록된 **탈퇴 사유와 시점에 나타나는 패턴을 탐색적으로 분석**하여, 사용자들이 어떤 이유로 서비스를 이탈하고 있는지 확인하고자 합니다.

- 이를 통해 서비스 이용 과정에서 발생할 수 있는 **불편 요소와 이탈에 영향을 미치는 요인을 간접적으로 파악**하고, 최종적으로 **사용자 이탈을 줄이기 위한 서비스 개선 방향을 도출하는 것**을 목표로 합니다.

In [4]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [5]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_userwithdraw`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

      id reason                created_at
0  47374  admin 2023-05-28 15:07:43+00:00
1  47376  admin 2023-05-29 06:22:53+00:00
2  47377  admin 2023-05-29 06:43:36+00:00
3  47379  admin 2023-05-29 08:33:47+00:00
4  47380  admin 2023-05-29 09:15:01+00:00


In [6]:
df.shape

(70764, 3)

## 전처리 수행을 위한 데이터 확인

In [7]:
# 결측값 확인
df.isnull().sum()

id            0
reason        0
created_at    0
dtype: int64

In [8]:
# 중복값 확인
df.duplicated().sum() #0
df[['reason', 'created_at']].duplicated().sum() # 369

np.int64(369)

In [9]:
df[df[['reason', 'created_at']].duplicated(keep=False)]

,id,reason,created_at
1113,476,기타 이유,2023-04-22 23:37:14+00:00
1114,477,기타 이유,2023-04-22 23:37:14+00:00
1969,2060,기타 이유,2023-04-30 10:42:13+00:00
1970,2061,기타 이유,2023-04-30 10:42:13+00:00
2061,2215,기타 이유,2023-04-30 13:46:47+00:00
...,...,...,...
65935,41490,함께 할 친구가 없어서,2023-05-23 14:06:20+00:00
66495,44108,함께 할 친구가 없어서,2023-05-25 08:19:49+00:00
66496,44109,함께 할 친구가 없어서,2023-05-25 08:19:49+00:00
66857,46001,함께 할 친구가 없어서,2023-05-26 11:31:24+00:00


In [10]:
df[df[['reason', 'created_at']].duplicated(keep=False)]['reason'].value_counts()

reason
기타 이유           571
함께 할 친구가 없어서    113
재밌는 질문이 없어서      48
버그가 너무 많아서        2
Name: count, dtype: int64

In [11]:
# 이상치 확인
df['reason'].value_counts()

reason
기타 이유           40301
함께 할 친구가 없어서    14450
재밌는 질문이 없어서     13133
버그가 너무 많아서       2031
구독료가 너무 비싸서       730
admin              61
test               53
기타                  5
Name: count, dtype: int64

In [12]:
display(df[df['reason'] == 'admin']['created_at'].min(), df[df['reason'] == 'admin']['created_at'].max())
display(df[df['reason'] == 'test']['created_at'].min(), df[df['reason'] == 'test']['created_at'].max())

Timestamp('2023-05-28 15:07:43+0000', tz='UTC')

Timestamp('2024-04-30 07:36:33+0000', tz='UTC')

Timestamp('2023-03-31 11:45:11+0000', tz='UTC')

Timestamp('2023-06-02 13:33:13+0000', tz='UTC')

In [13]:
display(df[df['reason'] == '기타']['created_at'].min(), df[df['reason'] == '기타']['created_at'].max())
display(df[df['reason'] == '기타 이유']['created_at'].min(), df[df['reason'] == '기타 이유']['created_at'].max())

Timestamp('2023-05-27 11:59:56+0000', tz='UTC')

Timestamp('2023-06-05 06:16:08+0000', tz='UTC')

Timestamp('2023-03-30 00:34:23+0000', tz='UTC')

Timestamp('2024-05-09 08:49:06+0000', tz='UTC')

## 전처리 수행 계획

### 1. `reason` 컬럼 정제

#### `test`, `admin` 데이터 제외

- `reason` 값이 **`test`, `admin`인 데이터는 분석 대상에서 제외**합니다.
- 해당 값은 기획서상 일반 사용자가 선택할 수 없는 값으로, 실제 사용자의 탈퇴보다는 **관리자 또는 내부 테스트 과정에서 생성된 데이터일 가능성이 높다**고 판단했습니다.
- 따라서 일반 사용자의 탈퇴 사유를 분석하는 본 분석의 목적과 맞지 않는 데이터로 판단하여 제외합니다.

#### `기타`, `기타 이유` 통합

- 기획서상 사용자가 선택할 수 있는 탈퇴 사유는 **`기타`**로 명시되어 있으나, 실제 데이터에는 `기타`와 `기타 이유`가 별도로 존재합니다.
- 두 값이 각각 **추가 사유를 작성하지 않은 경우와 주관식 사유를 작성한 경우**일 가능성은 있으나, 현재 데이터와 기획서만으로는 이를 명확하게 구분할 근거가 없습니다.
- 따라서 임의로 두 유형을 구분하지 않고, **`기타 이유`를 `기타`로 통합하여 동일한 탈퇴 사유로 처리**합니다.

### 2. 중복 이벤트 제거

- 전체 3개 컬럼을 기준으로는 완전히 동일한 행이 존재하지 않지만, **탈퇴 사유(`reason`)와 탈퇴 일시를 기준으로 중복 이벤트가 확인**됩니다.
- 사용자 식별 정보가 존재하지 않아 서로 다른 사용자가 동일한 시각에 같은 사유로 탈퇴했을 가능성을 완전히 배제할 수는 없습니다.
- 다만 여러 사용자가 **동일한 시각에 동일한 사유로 탈퇴했을 가능성은 상대적으로 낮다**고 판단했으며, 중복 데이터를 제거하더라도 전체 데이터 규모에 미치는 영향이 크지 않음을 확인했습니다.
- 이에 따라 해당 데이터를 **중복 이벤트로 간주하여 제거한 후 분석을 진행**합니다.

> **유의사항:** 사용자 식별 정보가 없는 데이터 구조의 한계로 인해 중복 여부를 확정할 수 없으므로, 해당 처리는 분석을 위한 전처리 가정에 해당합니다.

# 전처리 진행

In [14]:
del_sql = f"""
DELETE FROM `{PROJECT_ID}.{DATA_SET}.accounts_userwithdraw`
WHERE reason IN ('admin', 'test')
"""

query_job = client.query(del_sql)
query_job.result()  # DML 완료 대기

print(f"\n삭제 완료! 총 삭제된 행 수: {query_job.num_dml_affected_rows}건")


삭제 완료! 총 삭제된 행 수: 114건


In [17]:
update_sql = f"""
UPDATE `{PROJECT_ID}.{DATA_SET}.accounts_userwithdraw`
SET reason = CASE
    WHEN reason = '기타 이유' THEN '기타'
    ELSE reason
END
WHERE reason IS NOT NULL;
"""

query_job = client.query(update_sql)
query_job.result()  # DML 완료 대기

print(f"\n수정 완료! 총 수정된 행 수: {query_job.num_dml_affected_rows}건")


수정 완료! 총 수정된 행 수: 70650건


In [18]:
# 중복 데이터 삭제

delete_sql = f"""
DELETE FROM `{PROJECT_ID}.{DATA_SET}.accounts_userwithdraw`
WHERE id IN (
    SELECT id
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_userwithdraw`
    QUALIFY ROW_NUMBER() OVER(PARTITION BY reason, created_at ORDER BY id ASC) > 1
);
"""

query_job = client.query(delete_sql)
query_job.result()  # DML 완료 대기

print(f"\n삭제 완료! 총 삭제된 행 수: {query_job.num_dml_affected_rows}건")


삭제 완료! 총 삭제된 행 수: 369건


In [19]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_userwithdraw`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

      id       reason                created_at
0  69901  구독료가 너무 비싸서 2024-01-26 12:50:37+00:00
1  25824  구독료가 너무 비싸서 2023-05-15 10:37:40+00:00
2  44701  구독료가 너무 비싸서 2023-05-25 13:29:19+00:00
3  53418  구독료가 너무 비싸서 2023-06-15 08:15:47+00:00
4  23058  구독료가 너무 비싸서 2023-05-14 06:26:31+00:00


In [21]:
df['reason'].value_counts()

reason
기타              40019
함께 할 친구가 없어서    14393
재밌는 질문이 없어서     13109
버그가 너무 많아서       2030
구독료가 너무 비싸서       730
Name: count, dtype: int64